# Mobius → ONNX World Model: end-to-end

This notebook builds a tiny deterministic latent-dynamics model with Mobius, exports it to ONNX, loads it with `onnx-world-model`, and runs both stateless inference and a stateful rollout.

It is intentionally small: CPU-only, no model downloads, and normally finishes in a few seconds.

In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path


def find_runtime_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent):
        if (candidate / "python" / "onnx_world_model").is_dir():
            return candidate
    raise RuntimeError("Launch this notebook from onnx-world-model/ or notebooks/.")


RUNTIME_ROOT = find_runtime_root()
MOBIUS_ROOT = RUNTIME_ROOT.parent / "mobius"
if not (MOBIUS_ROOT / "src" / "mobius").is_dir():
    raise RuntimeError(f"Expected the Mobius checkout at {MOBIUS_ROOT}")

# Install the local checkouts only when the active notebook kernel does not
# already provide them. onnx-world-model builds its native extension here.
missing = [
    name
    for name in ("mobius", "onnx_world_model")
    if importlib.util.find_spec(name) is None
]
if missing:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-e",
            str(MOBIUS_ROOT),
            "-e",
            str(RUNTIME_ROOT),
        ]
    )

print("Mobius:", MOBIUS_ROOT)
print("onnx-world-model:", RUNTIME_ROOT)

## 1. Build the ONNX graph with Mobius

The model contract is:

`observation + action + state → next_state + observation_prediction + reward + continuation`

In [ ]:
import shutil

import torch
from torch import nn

from mobius import LatentDynamicsConfig, MLPLatentDynamicsModel, build_from_module


config = LatentDynamicsConfig(
    observation_shape=(2, 2),
    action_shape=(2,),
    state_shape=(3,),
    hidden_size=8,
    num_hidden_layers=2,
    residual_state=True,
)
package = build_from_module(
    MLPLatentDynamicsModel(config),
    config,
    task="latent-dynamics",
    execution_provider="cpu",
)


class TorchWeights(nn.Module):
    """A matching module used only to create a correctly named state dict."""

    def __init__(self, model_config: LatentDynamicsConfig) -> None:
        super().__init__()
        input_size = (
            model_config.observation_size
            + model_config.action_size
            + model_config.state_size
        )
        self.input_layer = nn.Linear(input_size, model_config.hidden_size)
        self.hidden_layers = nn.ModuleList(
            [nn.Linear(model_config.hidden_size, model_config.hidden_size)]
        )
        self.state_head = nn.Linear(model_config.hidden_size, model_config.state_size)
        self.observation_head = nn.Linear(
            model_config.hidden_size, model_config.observation_size
        )
        self.reward_head = nn.Linear(model_config.hidden_size, 1)
        self.continuation_head = nn.Linear(model_config.hidden_size, 1)


# Zero weights make the expected behavior obvious:
# next_state == state, observation/reward == 0, continuation == 0.5.
weights = TorchWeights(config)
for parameter in weights.parameters():
    nn.init.zeros_(parameter)
package.apply_weights(dict(weights.state_dict()))

EXPORT_DIR = RUNTIME_ROOT / "notebooks" / "artifacts" / "tiny_latent_dynamics"
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
package.save(str(EXPORT_DIR), progress_bar=False)

MODEL_PATH = EXPORT_DIR / "model.onnx"
assert MODEL_PATH.is_file()
print("Exported:", MODEL_PATH)
print("Size:", MODEL_PATH.stat().st_size, "bytes")

## 2. Load and run it with onnx-world-model

In [ ]:
import numpy as np

from onnx_world_model import LatentDynamicsModel


model = LatentDynamicsModel(MODEL_PATH, providers=["cpu"])
print("Inputs:", [(value.name, value.shape) for value in model.metadata.inputs])
print("Outputs:", [(value.name, value.shape) for value in model.metadata.outputs])

observation = np.arange(4, dtype=np.float32).reshape(1, 2, 2)
action = np.array([[0.25, -0.5]], dtype=np.float32)
state = np.array([[1.0, 2.0, 3.0]], dtype=np.float32)

result = model.step(observation, action, state)
np.testing.assert_allclose(result.next_state, state)
np.testing.assert_allclose(result.observation_prediction, 0.0)
np.testing.assert_allclose(result.reward, 0.0)
np.testing.assert_allclose(result.continuation, 0.5)

print("next_state:", result.next_state)
print("observation_prediction:\n", result.observation_prediction)
print("reward:", result.reward)
print("continuation:", result.continuation)

## 3. Stateful rollout

`Rollout` owns the recurrent state. The first step initializes it to zeros; later calls automatically feed the previous `next_state` back into the ONNX model.

In [ ]:
rollout = model.create_rollout()
rollout.reset(batch_size=1)

for step in range(3):
    step_result = rollout.step(observation, action)
    print(f"step {step}: state={step_result.next_state}, reward={step_result.reward.ravel()}")

np.testing.assert_allclose(rollout.state, np.zeros((1, 3), dtype=np.float32))
print("E2E succeeded: Mobius export and onnx-world-model inference both work.")